# 🐳 Docker do zero

### Mini curso prático · 2 horas · Windows e Mac

Ao final desta aula você vai saber **o que é** Docker, **por que** ele existe e **como usar** no dia a dia:
rodar contêineres prontos, criar a sua própria imagem com um `Dockerfile`, guardar dados em volumes,
ligar vários serviços com o Docker Compose — e até controlar tudo isso a partir do Python.

| Parte | Assunto | Tempo |
|---|---|---|
| 0 | Preparação e conceitos | 20 min |
| 1 | Primeiros contêineres: `run`, `ps`, `logs`, portas | 25 min |
| 2 | Configuração e dados: variáveis, `exec`, volumes | 20 min |
| 3 | Sua própria imagem: `Dockerfile` | 20 min |
| 4 | Redes e Docker Compose | 20 min |
| 5 | Docker pelo Python + o Weaviate do curso | 10 min |
| 6 | Limpeza e colinha | 5 min |

> **Antes da aula:** instale o [Docker Desktop](https://www.docker.com/products/docker-desktop/)
> (no Windows ele pede para ativar o **WSL2** — aceite e reinicie) e deixe-o **aberto**.
> Os passos estão no `README.md` desta pasta.

---
# Parte 0 — Preparação e conceitos (20 min)

## 0.1 Por que Docker existe

Todo mundo já ouviu (ou disse): *"mas na minha máquina funciona!"*.
O programa depende de **muita coisa além do código**: versão do Python, bibliotecas, programas do
sistema, variáveis de ambiente... Basta uma diferença para quebrar.

O Docker resolve isso empacotando **o programa junto com tudo de que ele precisa** num pacote
chamado **imagem**. Quem roda a imagem tem exatamente o mesmo ambiente — no seu notebook, no do
colega ou num servidor.

## 0.2 Máquina virtual × contêiner

![Máquina virtual × contêineres](diagramas/vm_vs_conteiner.svg)

- Uma **VM** emula um computador inteiro, com sistema operacional próprio: gigabytes e minutos para subir.
- Um **contêiner** é só um **processo isolado** que divide o kernel com o sistema: megabytes e
  segundos (ou menos) para subir.

### E no Windows e no Mac?

Contêineres usam recursos do **kernel Linux**. Windows e macOS não são Linux, então o
**Docker Desktop** roda uma **VM Linux bem leve** escondida:

| | Onde fica o Linux |
|---|---|
| 🪟 Windows | no **WSL2** (Subsistema do Windows para Linux) |
| 🍎 Mac | numa VM gerenciada pelo próprio Docker Desktop |

![O comando docker conversa com o Docker Engine, que roda numa VM Linux leve](diagramas/docker_desktop.svg)

Você não precisa mexer nessa VM. Os comandos `docker` que você digita conversam com ela.

## 0.3 Vocabulário

| Termo | O que é | Analogia |
|---|---|---|
| **Imagem** | pacote somente-leitura com o programa e suas dependências | a receita / o molde |
| **Contêiner** | uma imagem **em execução** (pode haver vários da mesma imagem) | o bolo assado |
| **Camada** | cada passo da construção da imagem; camadas iguais são reaproveitadas | |
| **Registry** | "loja" de imagens. O público principal é o [Docker Hub](https://hub.docker.com) | App Store |
| **Tag** | versão da imagem: `python:3.13-slim`, `nginx:latest` | |
| **Volume** | pasta gerenciada pelo Docker onde os dados sobrevivem ao contêiner | HD externo |

![Do Dockerfile à imagem e da imagem aos contêineres](diagramas/imagem_conteiner.svg)

Uma imagem dá origem a **quantos contêineres você quiser**, como um molde. Ela própria nunca muda:
cada contêiner escreve numa camada só dele, por cima.

## 0.4 Um ajudante para rodar comandos

Vamos rodar comandos `docker` de dentro do notebook. Em Jupyter dá para usar `!docker ...`,
mas o `!` usa o terminal do sistema: **cmd.exe no Windows** e **bash/zsh no Mac**, que têm
regras diferentes de aspas e variáveis. Para que **a mesma célula funcione nos dois**, usamos uma
pequena função `rodar()` com o módulo `subprocess` do Python.

> 💡 Tudo o que aparece dentro de `rodar("...")` você pode digitar **igualzinho** num terminal
> (PowerShell, Prompt de Comando ou Terminal do Mac).

In [1]:
import platform
import shlex
import subprocess
import time
from pathlib import Path

SISTEMA = platform.system()  # "Windows", "Darwin" (Mac) ou "Linux"
print(f"Sistema: {SISTEMA} · arquitetura do processador: {platform.machine()}")

# Pasta desta aula (onde está o notebook). Usamos pathlib para que os caminhos
# funcionem com "\" no Windows e "/" no Mac sem nos preocuparmos.
AQUI = Path.cwd()
EXEMPLOS = AQUI / "exemplos"
assert EXEMPLOS.exists(), f"Abra o notebook de dentro da pasta aulas/docker (estou em {AQUI})"


def rodar(comando: str, pasta: Path | None = None, mostrar: bool = True):
    """Roda um comando de terminal e mostra a saída. Funciona igual no Windows e no Mac.

    Com mostrar=False não imprime nada e devolve o resultado (stdout, returncode...).
    """
    if mostrar:
        print(f"$ {comando}")
    resultado = subprocess.run(
        shlex.split(comando),
        cwd=pasta,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    saida = (resultado.stdout + resultado.stderr).strip()
    if mostrar and saida:
        print(saida)
    if mostrar and resultado.returncode != 0:
        print(f"⚠️  o comando terminou com erro (código {resultado.returncode})")
    return None if mostrar else resultado


def esperar_http(url: str, segundos: int = 30) -> str:
    """Tenta acessar a URL até ela responder. Contêineres levam um instante para subir."""
    import requests

    for _ in range(segundos * 2):
        try:
            resposta = requests.get(url, timeout=2)
            resposta.encoding = "utf-8"  # o nginx não informa a codificação; sem isto "ã" vira "Ã£"
            return resposta.text
        except requests.ConnectionError:
            time.sleep(0.5)
    raise TimeoutError(f"{url} não respondeu em {segundos}s")


rodar("docker version")

Sistema: Darwin · arquitetura do processador: arm64
$ docker version
Client:
 Version:           29.6.2-rd
 API version:       1.54 (downgraded from 1.55)
 Go version:        go1.26.4
 Git commit:        ede120b
 Built:             Fri Jul 17 16:58:36 2026
 OS/Arch:           darwin/arm64
 Context:           rancher-desktop

Server:
 Engine:
  Version:          29.5.3
  API version:      1.54 (minimum version 1.41)
  Go version:       go1.26.3
  Git commit:       285b47192d4b2f183aba5dd360a92cd52d723004
  Built:            Sun Jun  7 14:14:36 2026
  OS/Arch:          linux/arm64
  Experimental:     false
 containerd:
  Version:          v2.3.2
  GitCommit:        fff62f14765df376e5fc36f5a8f8e795b5670f61
 runc:
  Version:          1.4.3
  GitCommit:        bb14dabeb7185bb72c8c86735d090dcb20f36587
 docker-init:
  Version:          0.19.0
  GitCommit:


Você deve ver duas seções: **Client** (o comando `docker`, no seu sistema) e **Server** (o
Docker Engine, dentro da VM Linux). Se aparecer *"Cannot connect to the Docker daemon"* ou
*"error during connect"*, o **Docker Desktop não está aberto**: abra-o, espere a baleia ficar
verde e rode a célula de novo.

Repare na linha `OS/Arch` do **Server**: é sempre `linux/...`, mesmo no Windows e no Mac. É a VM
de que falamos.

### Arquiteturas: `amd64` × `arm64`

Imagens são compiladas para um tipo de processador. Os PCs com Intel/AMD usam **amd64**; os Macs
com chip M1/M2/M3/M4 usam **arm64**. As imagens oficiais (python, nginx, redis...) vêm nas duas
versões, e o Docker baixa a certa sozinho.

In [2]:
r = rodar("docker version --format {{.Server.Os}}/{{.Server.Arch}}", mostrar=False)
print("O Docker Engine roda em:", r.stdout.strip())

O Docker Engine roda em: linux/arm64


---
# Parte 1 — Primeiros contêineres (25 min)

## 1.1 Olá, mundo

O comando mais importante é `docker run IMAGEM`. Ele:
1. procura a imagem no seu computador;
2. se não achar, **baixa** do Docker Hub (*pull*);
3. cria um contêiner a partir dela e o **executa**.

In [3]:
rodar("docker run hello-world")

$ docker run hello-world
Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/


O contêiner imprimiu a mensagem e **terminou**. Um contêiner vive enquanto o seu processo
principal estiver rodando — aqui era só um `print`.

## 1.2 Listando contêineres e imagens

| Comando | Mostra |
|---|---|
| `docker ps` | contêineres **rodando agora** |
| `docker ps -a` | **todos**, inclusive os que já terminaram |
| `docker images` | imagens baixadas |

In [4]:
rodar("docker ps")
print()
rodar("docker ps -a --filter ancestor=hello-world")
print()
rodar("docker images hello-world")

$ docker ps
CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

$ docker ps -a --filter ancestor=hello-world
CONTAINER ID   IMAGE         COMMAND    CREATED         STATUS                     PORTS     NAMES
25738fb5c03b   hello-world   "/hello"   3 seconds ago   Exited (0) 2 seconds ago             reverent_joliot

$ docker images hello-world
IMAGE                ID             DISK USAGE   CONTENT SIZE   EXTRA
hello-world:latest   5e2309035332       22.6kB         10.3kB   U


O `hello-world` aparece em `ps -a` com status `Exited (0)`: terminou com sucesso, mas o contêiner
**continua existindo** até ser apagado. Por isso é comum usar `--rm`, que apaga o contêiner
automaticamente quando ele termina:

In [5]:
# Apaga os hello-world que já terminaram: o `-q` lista só os IDs, que passamos ao `docker rm`.
# (O `docker container prune` parece mais direto, mas não aceita o filtro `ancestor`.)
ids = rodar("docker ps -aq --filter ancestor=hello-world", mostrar=False).stdout.split()
if ids:
    rodar("docker rm " + " ".join(ids))

# ...e roda um novo que se apaga sozinho. Repare: agora o download não acontece.
rodar("docker run --rm hello-world", mostrar=False)
rodar("docker ps -a --filter ancestor=hello-world")

$ docker rm 25738fb5c03b
25738fb5c03b
$ docker ps -a --filter ancestor=hello-world
CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


## 1.3 Um contêiner que fica rodando: servidor web

Agora algo mais útil: o **nginx**, um servidor web. Três opções novas:

| Opção | Significado |
|---|---|
| `-d` | *detached*: roda em segundo plano e devolve o terminal |
| `--name web-aula` | dá um nome ao contêiner (senão o Docker inventa um, como `quirky_turing`) |
| `-p 8088:80` | **publica a porta**: `porta_no_seu_computador:porta_no_contêiner` |

![Publicando portas com -p](diagramas/portas.svg)

Sem o `-p`, o nginx rodaria normalmente, mas **isolado**: nada de fora conseguiria falar com ele.

In [6]:
rodar("docker run -d --name web-aula -p 8088:80 nginx:alpine")
rodar("docker ps --filter name=web-aula")

$ docker run -d --name web-aula -p 8088:80 nginx:alpine
21b3534cfcf40c8cb8429fad3bb5c30e764bf2b4d5fd93971c258823cb683364
$ docker ps --filter name=web-aula
CONTAINER ID   IMAGE          COMMAND                  CREATED                  STATUS                  PORTS                                     NAMES
21b3534cfcf4   nginx:alpine   "/docker-entrypoint.…"   Less than a second ago   Up Less than a second   0.0.0.0:8088->80/tcp, [::]:8088->80/tcp   web-aula


In [7]:
html = esperar_http("http://localhost:8088")
print(html[:300])

<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
html { color-scheme: light dark; }
body { width: 35em; margin: 0 auto;
font-family: Tahoma, Verdana, Arial, sans-serif; }
</style>
</head>
<body>
<h1>Welcome to nginx!</h1>
<p>If you see this page, nginx is successfully installed


Abra também no seu navegador: **http://localhost:8088** (a célula abaixo faz isso por você).

In [8]:
import webbrowser

webbrowser.open("http://localhost:8088")

True

## 1.4 Logs, parar, iniciar, remover

O nginx registra cada acesso. `docker logs` mostra tudo o que o processo escreveu na tela.

In [9]:
rodar("docker logs --tail 5 web-aula")

$ docker logs --tail 5 web-aula
172.17.0.1 - - [01/Oct/2026:19:59:54 +0000] "GET /favicon.ico HTTP/1.1" 404 555 "-" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Code/1.140.0 Chrome/150.0.7871.250 Electron/43.7.3 Safari/537.36" "-"
172.17.0.1 - - [01/Oct/2026:20:00:01 +0000] "GET / HTTP/1.1" 200 896 "-" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36" "-"
172.17.0.1 - - [01/Oct/2026:20:00:01 +0000] "GET /favicon.ico HTTP/1.1" 404 555 "http://localhost:8088/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36" "-"
2026/10/01 19:59:54 [error] 30#30: *2 open() "/usr/share/nginx/html/favicon.ico" failed (2: No such file or directory), client: 172.17.0.1, server: localhost, request: "GET /favicon.ico HTTP/1.1", host: "localhost:8088"
2026/10/01 20:00:01 [error] 32#32: *4 open() "/usr/share/nginx/html/favicon.ico"

In [10]:
rodar("docker stop web-aula")      # para o processo (o contêiner continua existindo)
rodar("docker ps -a --filter name=web-aula --format \"{{.Names}}: {{.Status}}\"")

rodar("docker start web-aula")     # liga de novo, com a mesma configuração
rodar("docker ps -a --filter name=web-aula --format \"{{.Names}}: {{.Status}}\"")

$ docker stop web-aula
web-aula
$ docker ps -a --filter name=web-aula --format "{{.Names}}: {{.Status}}"
web-aula: Exited (137) Less than a second ago
$ docker start web-aula
web-aula
$ docker ps -a --filter name=web-aula --format "{{.Names}}: {{.Status}}"
web-aula: Up Less than a second


Vamos deixar o `web-aula` rodando por enquanto: ele será útil na Parte 2.

**Ciclo de vida de um contêiner:**

![Ciclo de vida: run, stop, start, rm](diagramas/ciclo_de_vida.svg)

## 🎯 Exercício 1

1. Suba um **segundo** nginx chamado `web2` na porta **8089** e confirme que os dois aparecem em `docker ps`.
2. Tente subir um terceiro na porta **8088**. Que erro aparece? Por quê?
3. Pare e remova o `web2`.

<details>
<summary>👀 Ver solução</summary>

```python
rodar("docker run -d --name web2 -p 8089:80 nginx:alpine")
rodar("docker ps")

# Erro "port is already allocated": só um processo pode usar
# uma porta do seu computador de cada vez. A porta de DENTRO (80)
# pode repetir à vontade: cada contêiner tem a sua.
rodar("docker run -d --name web3 -p 8088:80 nginx:alpine")
rodar("docker rm web3")  # ele foi criado, mas não subiu

rodar("docker rm -f web2")  # -f = para e remove de uma vez
```
</details>

In [ ]:
# Sua resposta aqui

---
# Parte 2 — Configuração e dados (20 min)

## 2.1 Variáveis de ambiente (`-e`)

A forma padrão de **configurar** um contêiner é por variáveis de ambiente. A mesma imagem se
comporta diferente conforme o que você passa em `-e NOME=valor`.

Aqui usamos o **alpine**, um Linux minúsculo (~8 MB), e mandamos executar um comando dentro dele:

In [11]:
rodar("docker run --rm -e NOME=Turma alpine sh -c 'echo Olá, $NOME! Eu sou um Linux dentro do $(uname -s)'")

$ docker run --rm -e NOME=Turma alpine sh -c 'echo Olá, $NOME! Eu sou um Linux dentro do $(uname -s)'
Olá, Turma! Eu sou um Linux dentro do Linux


> 💡 Tudo depois do nome da imagem (`sh -c '...'`) substitui o comando padrão da imagem. E quem
> interpreta o `$NOME` é o `sh` **do contêiner** — por isso funciona igual no Windows e no Mac.

## 2.2 Entrando num contêiner que está rodando (`exec`)

`docker exec` executa um comando **dentro** de um contêiner que já está de pé. É a principal
ferramenta para investigar problemas.

In [12]:
rodar("docker exec web-aula cat /etc/os-release")
print()
rodar("docker exec web-aula ls /usr/share/nginx/html")

$ docker exec web-aula cat /etc/os-release
NAME="Alpine Linux"
ID=alpine
VERSION_ID=3.24.2
PRETTY_NAME="Alpine Linux v3.24"
HOME_URL="https://alpinelinux.org/"
BUG_REPORT_URL="https://gitlab.alpinelinux.org/alpine/aports/-/issues"

$ docker exec web-aula ls /usr/share/nginx/html
50x.html
index.html


> 💡 **No terminal**, você pode abrir um shell interativo dentro do contêiner com
> `docker exec -it web-aula sh` (saia com `exit`). O `-it` não funciona dentro do notebook,
> porque o notebook não é um terminal interativo.

## 2.3 O problema: contêineres são descartáveis

Tudo o que é escrito **dentro** de um contêiner some quando ele é removido. Vamos provar:

In [13]:
rodar("docker exec web-aula sh -c 'echo \"<h1>Editado dentro do contêiner</h1>\" > /usr/share/nginx/html/index.html'")
print(esperar_http("http://localhost:8088"))

rodar("docker rm -f web-aula")
rodar("docker run -d --name web-aula -p 8088:80 nginx:alpine")
print(esperar_http("http://localhost:8088")[:120], "...")

$ docker exec web-aula sh -c 'echo "<h1>Editado dentro do contêiner</h1>" > /usr/share/nginx/html/index.html'
<h1>Editado dentro do contêiner</h1>

$ docker rm -f web-aula
web-aula
$ docker run -d --name web-aula -p 8088:80 nginx:alpine
7587daadb92cded1ab6686f5f37003b859fb9066b0b38e29d661a1b8b6b24a8e
<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
html { color-scheme: light dark; }
body { width:  ...


A edição sumiu: o contêiner novo nasceu da imagem original. Isso é **bom** (todo contêiner começa
limpo e previsível), mas e quando precisamos guardar dados — um banco de dados, por exemplo?

## 2.4 As duas soluções: bind mount e volume

![Bind mount × volume](diagramas/bind_mount_e_volume.svg)

### Bind mount: o nginx servindo uma pasta do seu computador

A pasta `exemplos/site` tem um `index.html`. Vamos "emprestá-la" ao nginx.

> 🪟🍎 O caminho completo da pasta é diferente em cada máquina (`C:\Users\...` no Windows,
> `/Users/...` no Mac). O `pathlib` monta o caminho certo, e o `.as_posix()` troca `\` por `/`, que
> o Docker aceita nos dois sistemas.

In [ ]:
pasta_site = (EXEMPLOS / "site").resolve().as_posix()
print("Pasta montada:", pasta_site)

rodar("docker rm -f web-aula", mostrar=False)
rodar(f"docker run -d --name web-aula -p 8088:80 -v \"{pasta_site}:/usr/share/nginx/html:ro\" nginx:alpine")
print(esperar_http("http://localhost:8088"))

Abra **http://localhost:8088** no navegador, edite `exemplos/site/index.html` no seu editor,
salve e recarregue a página: a mudança aparece na hora. O `:ro` no fim (*read-only*) impede que
o contêiner altere os seus arquivos.

### Volume: dados que sobrevivem ao contêiner

Criamos um volume, escrevemos um arquivo nele com um contêiner, **apagamos o contêiner** e lemos
o arquivo com outro contêiner novinho.

In [ ]:
rodar("docker volume create dados-aula")

# Contêiner 1: escreve e morre (--rm)
rodar("docker run --rm -v dados-aula:/dados alpine sh -c 'date > /dados/criado_em.txt; echo gravado!'")

# Contêiner 2: outro contêiner, mesmo volume
rodar("docker run --rm -v dados-aula:/dados alpine cat /dados/criado_em.txt")

rodar("docker volume ls --filter name=dados-aula")

## 🎯 Exercício 2

1. Rode um contêiner `alpine` que **acrescente** (`>>`) uma linha com a data em `/dados/historico.txt`
   no volume `dados-aula`. Rode a célula 3 vezes.
2. Leia o arquivo com outro contêiner. Quantas linhas há?
3. Descubra a versão do Python que vem na imagem `python:3.13-slim` sem instalar Python nenhum,
   só com `docker run`.

<details>
<summary>👀 Ver solução</summary>

```python
rodar("docker run --rm -v dados-aula:/dados alpine sh -c 'date >> /dados/historico.txt'")
rodar("docker run --rm -v dados-aula:/dados alpine cat /dados/historico.txt")

rodar("docker run --rm python:3.13-slim python --version")
```
</details>

In [ ]:
# Sua resposta aqui

---
# Parte 3 — Sua própria imagem: o `Dockerfile` (20 min)

Até aqui só usamos imagens prontas. Agora vamos empacotar **o nosso próprio programa**: um
pequeno site em Python (Flask) que está em `exemplos/app_flask`.

```
exemplos/app_flask/
├── app.py             ← o programa
├── requirements.txt   ← as bibliotecas de que ele precisa
├── Dockerfile         ← a receita da imagem
└── .dockerignore      ← o que NÃO copiar para a imagem
```

In [ ]:
APP = EXEMPLOS / "app_flask"
print((APP / "Dockerfile").read_text(encoding="utf-8"))

### Lendo a receita

| Instrução | O que faz |
|---|---|
| `FROM` | imagem de partida. Quase sempre uma imagem oficial (`python`, `node`, `nginx`...) |
| `WORKDIR` | pasta de trabalho dentro da imagem; os comandos seguintes rodam nela |
| `COPY origem destino` | copia arquivos do seu computador para a imagem |
| `RUN` | executa um comando **durante a construção** (instalar coisas) |
| `EXPOSE` | documenta a porta em que o app escuta (não publica nada sozinho: isso é o `-p`) |
| `ENV` | define uma variável de ambiente padrão |
| `CMD` | o comando que roda **quando o contêiner sobe** |

> ⚠️ **`RUN` × `CMD`**: o `RUN` roda **uma vez**, ao construir a imagem. O `CMD` roda **toda vez**
> que um contêiner é criado a partir dela.

### Construindo (`docker build`)

`-t aula-flask:1.0` dá nome e tag à imagem. O `.` no final é o **contexto**: a pasta cujos
arquivos o `COPY` pode enxergar.

In [ ]:
inicio = time.time()
rodar("docker build -t aula-flask:1.0 .", pasta=APP)
print(f"\n⏱️  {time.time() - inicio:.1f} s")

### O cache de camadas

Cada instrução vira uma **camada**. Se nada mudou numa instrução nem nas anteriores, o Docker
**reaproveita** a camada do cache:

![Cache de camadas: o que é refeito no segundo build](diagramas/cache_de_camadas.svg)

Rode o build de novo:

In [ ]:
inicio = time.time()
rodar("docker build -t aula-flask:1.0 .", pasta=APP)
print(f"\n⏱️  {time.time() - inicio:.1f} s  ← repare nos CACHED")

É por isso que o `Dockerfile` copia o `requirements.txt` **antes** do resto do código: mudar o
`app.py` (o que acontece o tempo todo) não obriga a reinstalar as bibliotecas (o que é lento).

Dá para ver as camadas e o tamanho de cada uma com `docker history`:

In [ ]:
rodar("docker history aula-flask:1.0 --format \"{{.Size}}\t{{.CreatedBy}}\"")

### Rodando a nossa imagem

O app escuta na porta 5000 **dentro** do contêiner. Do lado de fora usamos a **8000**.

> 🍎 **Por que não 5000 dos dois lados?** No Mac, a porta 5000 já é usada pelo *AirPlay
> Receiver* do sistema, e o `-p 5000:5000` daria conflito (ou, pior, você falaria com o AirPlay
> sem perceber). É um bom lembrete: **a porta de fora é sua escolha**.

In [ ]:
rodar("docker run -d --name flask-aula -p 8000:5000 -e NOME=Turma aula-flask:1.0")
print(esperar_http("http://localhost:8000"))

O `Contêiner:` mostrado é o **hostname** do contêiner, que o Docker define como o ID dele. Compare
com o `docker ps`:

In [ ]:
rodar("docker ps --filter name=flask-aula --format \"{{.ID}}  {{.Names}}  {{.Ports}}\"")

## 🎯 Exercício 3

1. Mude a mensagem `Olá, {NOME}!` no `app.py` para outra coisa.
2. Reconstrua como `aula-flask:1.1` e observe quais passos vêm do cache.
3. Suba a versão nova na porta 8001, **sem derrubar** a 1.0, e compare as duas.

<details>
<summary>👀 Ver solução</summary>

```python
rodar("docker build -t aula-flask:1.1 .", pasta=APP)  # só o "COPY . ." em diante é refeito
rodar("docker run -d --name flask-11 -p 8001:5000 aula-flask:1.1")
print(esperar_http("http://localhost:8000"))
print(esperar_http("http://localhost:8001"))
rodar("docker rm -f flask-11")
```
</details>

In [ ]:
# Sua resposta aqui

---
# Parte 4 — Redes e Docker Compose (20 min)

## 4.1 Contêineres conversando entre si

Num app de verdade há várias peças: o site, o banco de dados, um cache... Cada uma no seu contêiner.
Para que conversem, colocamos todos numa mesma **rede**. Dentro dela, **o nome do contêiner
funciona como endereço**.

In [ ]:
rodar("docker network create rede-aula")
rodar("docker run -d --name meu-redis --network rede-aula redis:8-alpine")
time.sleep(2)

# Outro contêiner, na mesma rede, falando com o primeiro PELO NOME
rodar("docker run --rm --network rede-aula redis:8-alpine redis-cli -h meu-redis ping")
rodar("docker run --rm --network rede-aula redis:8-alpine redis-cli -h meu-redis set curso Docker")
rodar("docker run --rm --network rede-aula redis:8-alpine redis-cli -h meu-redis get curso")

Funciona, mas repare quantos comandos e opções já precisamos lembrar para **dois** contêineres.
Imagine dez. É aí que entra o Compose.

## 4.2 Docker Compose: tudo num arquivo só

O **Docker Compose** descreve todos os serviços, portas, variáveis, volumes e redes num arquivo
YAML. Um comando sobe tudo; outro derruba tudo.

Nosso exemplo junta o app Flask com um Redis que conta as visitas:

In [ ]:
COMPOSE = EXEMPLOS / "compose"
print((COMPOSE / "docker-compose.yml").read_text(encoding="utf-8"))

![Docker Compose: app e Redis na mesma rede](diagramas/compose.svg)

Pontos a reparar:
- `build: ../app_flask` → o Compose constrói a imagem com o nosso `Dockerfile`.
- `REDIS_HOST: redis` → o app acha o banco pelo **nome do serviço**. O Compose cria uma rede
  sozinho e põe os dois nela.
- `depends_on` + `healthcheck` → o app só sobe **depois** que o Redis responder.
- `volumes: redis_data` → as visitas sobrevivem se o Redis for recriado.

### Os comandos

| Comando | O que faz |
|---|---|
| `docker compose up -d --build` | constrói o que precisar e sobe tudo em segundo plano |
| `docker compose ps` | estado de cada serviço |
| `docker compose logs SERVIÇO` | logs de um serviço |
| `docker compose down` | para e remove contêineres e rede (os volumes ficam) |
| `docker compose down -v` | idem, **apagando os volumes** |

Primeiro liberamos a porta 8000, que o `flask-aula` da Parte 3 está usando:

In [ ]:
rodar("docker rm -f flask-aula", mostrar=False)
rodar("docker compose up -d --build", pasta=COMPOSE)
rodar("docker compose ps", pasta=COMPOSE)

In [ ]:
for _ in range(3):
    print(esperar_http("http://localhost:8000"))

Agora o teste de fogo: derrubar tudo **sem** `-v` e subir de novo. O contador continua de onde parou?

In [ ]:
rodar("docker compose down", pasta=COMPOSE)
rodar("docker compose up -d", pasta=COMPOSE)
print(esperar_http("http://localhost:8000"))

## 🎯 Exercício 4

1. Veja os logs só do serviço `app`.
2. Mude o `NOME` no `docker-compose.yml` e rode `docker compose up -d` de novo. O Compose recria
   **só** o serviço que mudou — confira na saída.
3. Derrube com `down -v` e suba de novo. O que aconteceu com o contador?

<details>
<summary>👀 Ver solução</summary>

```python
rodar("docker compose logs app", pasta=COMPOSE)
rodar("docker compose up -d", pasta=COMPOSE)    # "Recreated" só no app
rodar("docker compose down -v", pasta=COMPOSE)  # -v apaga o volume redis_data...
rodar("docker compose up -d", pasta=COMPOSE)
print(esperar_http("http://localhost:8000"))     # ...e o contador volta a 1
```
</details>

In [ ]:
# Sua resposta aqui

---
# Parte 5 — Docker pelo Python + o Weaviate do curso (10 min)

## 5.1 O SDK oficial

Tudo o que fizemos com o comando `docker` dá para fazer direto do Python com o pacote
[`docker`](https://docker-py.readthedocs.io). Útil para testes automatizados, scripts de
infraestrutura ou para subir um banco temporário dentro de um programa.

> O SDK precisa saber **onde** está o Docker Engine. Normalmente `docker.from_env()` acha sozinho;
> a função abaixo também pergunta ao comando `docker` qual é o endereço, para funcionar em
> qualquer instalação (Windows, Mac, Docker Desktop, Colima...).

In [ ]:
%pip install -q "docker>=7.1" requests

In [ ]:
import docker


def conectar_docker() -> docker.DockerClient:
    try:
        cliente = docker.from_env()
        cliente.ping()
        return cliente
    except docker.errors.DockerException:
        # Pergunta ao CLI qual endereço o "contexto" atual usa e conecta nele.
        r = rodar("docker context inspect --format {{.Endpoints.docker.Host}}", mostrar=False)
        return docker.DockerClient(base_url=r.stdout.strip())


cliente = conectar_docker()
print("Conectado ao Docker", cliente.version()["Version"])

for c in cliente.containers.list():
    print(f"  {c.name:25} {c.status:10} {c.image.tags}")

In [ ]:
# Rodar um contêiner e pegar a saída, como fizemos com `docker run --rm`
saida = cliente.containers.run("alpine", "echo Olá do SDK!", remove=True)
print(saida.decode())

# Subir um serviço, usar e jogar fora
redis_temp = cliente.containers.run("redis:8-alpine", detach=True, name="redis-sdk")
time.sleep(1)
print(redis_temp.exec_run("redis-cli ping").output.decode())
redis_temp.remove(force=True)
print("removido:", redis_temp.name)

## 5.2 Lendo um `docker-compose.yml` de verdade

No curso principal (`curso_agente_leis_local_docker.ipynb`), o banco vetorial **Weaviate** sobe
com o `docker-compose.yml` da raiz do repositório. Agora você consegue ler cada linha dele:

In [14]:
print((AQUI.parent.parent / "docker-compose.yml").read_text(encoding="utf-8"))

services:
  weaviate:
    image: cr.weaviate.io/semitechnologies/weaviate:1.39.5
    container_name: weaviate-curso
    ports:
      - "8080:8080"    # REST / GraphQL
      - "50051:50051"  # gRPC (o cliente Python 4.x usa os dois)
    restart: on-failure
    environment:
      # --- acesso ---
      # Aula local, sem autenticação. NUNCA use assim em produção.
      AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED: "true"

      # --- dados ---
      # Persistidos no volume abaixo: pode parar e subir de novo sem reimportar.
      PERSISTENCE_DATA_PATH: /var/lib/weaviate

      # ⚠️ NÃO REMOVA. Sem isto o nome do nó vira o ID do contêiner, que muda a cada
      # "docker compose up" que recria o contêiner. Como o estado do RAFT gravado no
      # volume guarda o nome antigo, o nó novo não reconhece o próprio cluster e o
      # Weaviate fica preso em "leader not found" — com os dados intactos e inacessíveis.
      CLUSTER_HOSTNAME: node1
      RAFT_BOOTSTRAP_EXPECT: 1

      # --- vetorização --

| Trecho | O que você já sabe |
|---|---|
| `image: cr.weaviate.io/...:1.39.5` | imagem de outro *registry* (não o Docker Hub), com **tag fixa** para todos terem a mesma versão |
| `container_name` | o `--name` |
| `ports: 8080`, `50051` | duas portas publicadas: REST e gRPC |
| `restart: on-failure` | se o processo cair com erro, o Docker o reinicia sozinho |
| `environment:` | a configuração toda é por variáveis de ambiente (`-e`) |
| `volumes: weaviate_data:` | **volume nomeado**: os dados importados sobrevivem a `down` |
| `healthcheck` | como o Docker sabe que o serviço está pronto (o `wget` existe na imagem; o `curl`, não) |

E o comentário sobre `CLUSTER_HOSTNAME` mostra uma armadilha real de contêineres: o **hostname
muda** a cada contêiner recriado (lembra do ID que o nosso Flask mostrava?). Se o programa grava
o próprio nome junto com os dados, ele deixa de se reconhecer. Fixar o hostname resolve.

---
# Parte 6 — Limpeza e colinha (5 min)

Imagens e volumes ocupam disco. Veja quanto:

In [ ]:
rodar("docker system df")

In [ ]:
# Remove tudo o que ESTA aula criou (e nada além disso)
rodar("docker compose down -v", pasta=COMPOSE)

# Contêineres, rede, volume e imagens das partes 1 a 3 (inclusive dos exercícios).
# Com mostrar=False, o que já tinha sido removido não gera mensagem de erro.
for comando in [
    "docker rm -f web-aula web2 web3 meu-redis flask-aula flask-11",
    "docker network rm rede-aula",
    "docker volume rm dados-aula",
    "docker rmi aula-flask:1.0 aula-flask:1.1 compose-app",
]:
    rodar(comando, mostrar=False)

rodar("docker ps -a --filter name=aula")
print("🧹 Limpo!")

> ⚠️ Existe também o `docker system prune`, que apaga **todos** os contêineres parados, redes sem
> uso e imagens órfãs do computador — inclusive de outros projetos. Útil, mas leia a pergunta de
> confirmação antes de dizer `y`. Com `-a --volumes` ele apaga ainda mais, incluindo **dados**.

## 📋 Colinha

| Quero... | Comando |
|---|---|
| rodar uma imagem | `docker run -d --name NOME -p FORA:DENTRO -e VAR=valor IMAGEM` |
| rodar e apagar ao terminar | `docker run --rm IMAGEM comando` |
| ver o que está rodando | `docker ps` (`-a` para todos) |
| ver logs | `docker logs -f NOME` |
| entrar no contêiner | `docker exec -it NOME sh` |
| parar / iniciar / remover | `docker stop` / `docker start` / `docker rm -f` |
| construir imagem | `docker build -t nome:tag .` |
| baixar imagem | `docker pull IMAGEM:tag` |
| guardar dados | `-v volume:/caminho` ou `-v /pasta/local:/caminho` |
| subir um projeto | `docker compose up -d --build` |
| derrubar um projeto | `docker compose down` (`-v` apaga dados) |
| ver espaço em disco | `docker system df` |

## 🎯 Desafio final

Escreva um `Dockerfile` para um script Python seu (qualquer um!) e rode-o com `docker run --rm`.
Bônus: crie um `docker-compose.yml` que suba o seu script junto com um `postgres:17` e passe a
senha do banco por variável de ambiente.

## Para continuar estudando

- [Documentação oficial do Docker](https://docs.docker.com/get-started/)
- [Referência do Dockerfile](https://docs.docker.com/reference/dockerfile/)
- [Referência do Compose](https://docs.docker.com/reference/compose-file/)
- [Play with Docker](https://labs.play-with-docker.com/) — Docker no navegador, sem instalar nada